# <i><u>Animation notebook</u></i>

This notebook provides the code to generate animations that illustrate the work <i>Revisiting the dynamical properties of the Pedlosky’s Two-Layer Model for Finite Amplitude Baroclinic Waves</i>. These animtions are used in the <code>README.md</code> file. We begin this notebook by importing the necessary packages and modules.

In [1]:
from PedloskySystem import System

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap

## <span style="color:orange">1.  Full model trajectory animation</span>

We first create the system and integrate it to get the trajectory data.

In [2]:
# Parameters
kc = 20
gamma = 0.17
a = np.pi * np.sqrt(2)
m = 1

# Initial condition construction
A0 = 1.
B0 = 0.

X0 = np.empty((1, kc + 2))
X0[0][0] = A0
X0[0][1] = B0

for i in range(2, len(X0[0])):
    X0[0][i] = - A0 ** 2

# Time integration
tMax = 10 ** 3
dt = 10 ** (- 1)
time_integration = np.linspace(0, tMax, int(tMax / dt))   

ds = System(kc, gamma, a, m)
solution = ds.integration_system(time_integration, X0)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


We now create the backbone of the animation.

In [3]:
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",})

fig, ax = plt.subplots(figsize = (10, 10 / 1.5))
ax.grid(True, linestyle = '-.', which = 'both')
ax.tick_params(axis = 'both', direction = 'in', length = 6, width = 1.5, labelsize = 15, grid_color = 'grey', grid_alpha = 0.5)
ax.set_title("Full model trajectory", fontsize = 20, pad = 20)

# These artists will be updated in the animation
point, = ax.plot([], [], 'o', color = 'navy', markersize = 4, zorder = 3)
xlabel = ax.set_xlabel(r"$A(0)$", fontsize = 20)
ylabel = ax.set_ylabel(r"$B(0)$", fontsize = 20)

# Extracting the x and y data for setting the limits of the plot
x_data = solution[0, :, 0]
y_data = solution[0, :, 1]
ax.set_xlim(np.min(x_data) - .25, np.max(x_data) + .25)
ax.set_ylim(np.min(y_data) - .25, np.max(y_data) + .25)

cmap = plt.get_cmap("PuBu")
lc = LineCollection([], cmap = cmap, linewidth = 1.5)
ax.add_collection(lc)

plt.close() # Prevents duplicate static plot in some environments

In [4]:
def update(frame):
    """"
    This function is called at each frame of the animation. It updates the line segments, the point, and the axis labels to reflect the current state of the trajectory.

    Parameters
    ----------
    frame : int
        The current frame number in the animation

    Return
    ------
    tuple
        A tuple containing the updated line collection, x-axis label, y-axis label, and point for the current frame

    Example
    -------
    See below for an example of usage
    """
    if frame < 2: 
        return lc, xlabel, ylabel, point  # Skip the first frame to avoid issues with segments
    
    points = np.array([x_data[:frame], y_data[:frame]]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis = 1)

    lc.set_segments(segments) # Update the line segments for the current frame
    lc.set_array(np.linspace(0, 1, len(segments))) # Set the color of the segments based on their position in the trajectory

    xlabel.set_text(rf"$A({int(time_integration[frame])})$")
    ylabel.set_text(rf"$B({int(time_integration[frame])})$")
    point.set_data([x_data[frame]], [y_data[frame]])
    
    return lc, xlabel, ylabel, point

In [5]:
frames = np.append(np.arange(0, len(x_data), 7), len(x_data) - 1)  # Ensure the last frame is included
ani = FuncAnimation(fig, update, frames = frames, interval = 20, blit = True)
ani.save('trajectory_animation.gif', writer = 'pillow')

## <span style="color:orange">2.  Toy model basins of attraction animation</span>

<u>Remark</u>: The following code requires precomputed basins of attraction for the toy model (with fixed values of $a$ and $m$, taken to be $\pi \sqrt{2}$ and $1$, respectively, as usual) for various values of $\gamma$, here ranging from $\gamma = 0.25$ to $0.45$ in steps of $10^{-3}$. The basins should cover the ranges $A_0, B_0 \in [-5, 5]$ with steps of $10^{-2}$. The resulting data should be stored, for example, as <code>basin_gamma_0.250.txt</code>, and is not provided here. Note that these are requirements for the following code to work — the user is free to adapt them as desired.

In [6]:
gamma_values = np.linspace(0.45, 0.25, 201)
result_basins = np.zeros((len(gamma_values), 1001, 1001)) # Preallocate the array to store the results of all basins for each gamma value

for i, gamma in enumerate(gamma_values):
    result_basins[i] = np.loadtxt(f"animation_data/basin_gamma_{gamma:.3f}.txt").reshape(1001, 1001)

We now create the backbone of the animation.

In [7]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.tick_params(axis = 'both', direction = 'in', length = 6, width = 1.5, labelsize = 15, grid_color = 'grey', grid_alpha = 0.5)
ax.set_xlabel(r"$A_0$", fontsize = 20)
ax.set_ylabel(r"$B_0$", fontsize = 20)

# This artist will be updated in the animation
text = ax.set_title(f"Toy model basins of attraction for $\gamma={gamma_values[0]:.3f}$", fontsize = 20, pad = 20)

# Defining a custom colormap
colors = ["white", "lightgray", "steelblue", "navy"]
custom_cmap = LinearSegmentedColormap.from_list("gray_to_blue", colors)

# Display the initial basin of attraction with the custom colormap
im = ax.imshow(result_basins[0], origin = 'lower', cmap = custom_cmap, extent = [-5, 5, -5, 5], interpolation = 'nearest', vmin = result_basins.min(), vmax = result_basins.max())
cbar = fig.colorbar(im, ax = ax, shrink = 0.6, pad = 0.05)
cbar.set_label(r'$A(10^4)$', fontsize = 18, labelpad = 10)
cbar.ax.tick_params(labelsize = 12)

plt.close()

In [8]:
def update(frame):
    """"
    This function is called at each frame of the animation. It updates the image data and the title text to reflect the current state of the basins of attraction for the given gamma value.

    Parameters
    ----------
    frame : int
        The current frame number in the animation

    Return
    ------
    tuple
        A tuple containing the updated image and title text for the current frame

    Example
    -------
    See below for an example of usage
    """
    im.set_data(result_basins[frame]) 
    text.set_text(f"Toy model basins of attraction for $\gamma={gamma_values[frame]:.3f}$")

    return im, text

In [9]:
frames = np.arange(len(gamma_values))
ani = FuncAnimation(fig, update, frames = frames, interval = 50, blit = True)
ani.save('basin_animation.gif', writer = 'pillow')